# 🧠 Aula 01 — Introdução às Arquiteturas de Computadores e GPUs

**Objetivo:** entender as arquiteturas **Von Neumann** e **Harvard** e por que a **GPU**
é a peça-chave da IA — e, ao final, **ver e medir** a diferença entre CPU e GPU.

**Roteiro deste notebook:**
1. Verificação do ambiente (tem GPU?).
2. Teoria: Von Neumann vs. Harvard.
3. Teoria: CPU vs. GPU.
4. Demo: consultar a GPU com `nvidia-smi`.
5. Atividade: somar muitos números — sequencial vs. vetorizado.
6. Discussão e síntese.

> 💡 **Não tem GPU?** Sem problema: o notebook detecta e gera dados **simulados** no
> mesmo formato, então a aula roda do começo ao fim.

## 1. Verificação do Ambiente

Antes de qualquer coisa, descobrimos se o ambiente tem uma GPU NVIDIA e o utilitário
`nvidia-smi`. Isso define se usaremos dados **reais** ou **simulados**.

In [ ]:
# @title 🔍 Detectar GPU e nvidia-smi no ambiente
# ============================================================================
# OBJETIVO: descobrir se há GPU NVIDIA disponível e o utilitário nvidia-smi.
# O resultado decide se usaremos dados REAIS ou SIMULADOS nas próximas células.
# ============================================================================
import shutil       # shutil.which() localiza um executável no PATH
import subprocess   # subprocess executa comandos do sistema operacional

# shutil.which devolve o caminho do executável, ou None se não existir
CAMINHO_NVIDIA_SMI = shutil.which("nvidia-smi")
TEM_GPU = CAMINHO_NVIDIA_SMI is not None   # True = podemos medir a GPU de verdade

if TEM_GPU:
    print(f"✅ nvidia-smi encontrado em: {CAMINHO_NVIDIA_SMI}")
    # Consulta rápida: nome da GPU, versão do driver e VRAM total
    info = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,driver_version,memory.total",
         "--format=csv,noheader"],
        capture_output=True, text=True,
    ).stdout
    print(info)
else:
    print("⚠️  nvidia-smi NÃO encontrado — a aula seguirá em MODO SIMULADO.")
    print("   Para dados reais: Runtime ➔ Change runtime type ➔ T4 GPU.")

## 2. Teoria: Von Neumann vs. Harvard

Um computador precisa buscar **instruções** (o que fazer) e **dados** (com o que fazer).
As duas arquiteturas clássicas diferem em **onde** essas coisas ficam:

| | Von Neumann | Harvard |
| :--- | :--- | :--- |
| Memória | **Única** para dados e instruções | **Separadas** |
| Acesso | Instrução e dado disputam o barramento | **Simultâneo** |
| Vantagem | Simples e barata | Mais rápida por ciclo |
| Onde aparece | PCs e servidores | Embarcados (microcontroladores, DSPs) |

O limite da Von Neumann é o **gargalo do barramento**: como os dois passam pelo mesmo
caminho, o processador às vezes **espera**. Caches e hierarquia de memória (Aula 3)
amenizam esse gargalo.

## 3. Teoria: CPU vs. GPU

- **CPU — o especialista:** poucos núcleos (4–64), alta frequência, ótima em tarefas
  **sequenciais** e lógica de controle (sistema operacional, decisões, I/O).
- **GPU — o exército:** milhares de núcleos, altíssima largura de banda de memória,
  feita para **paralelismo massivo** — ideal para **matrizes** e deep learning.

> **Analogia:** a CPU é um chef experiente que faz um prato complexo sozinho. A GPU são
> mil cozinheiros fazendo o mesmo prato simples ao mesmo tempo.

As próximas células mostram isso **na prática**.

## 4. Demo: consultar a GPU com `nvidia-smi`

O `nvidia-smi` é o utilitário oficial da NVIDIA. Ele mostra modelo, memória, temperatura
e processos da placa. No terminal, o comando é simplesmente:

```bash
!nvidia-smi
```

A célula abaixo pede apenas os campos que interessam, em CSV — formato ideal para juntar
com um *timestamp* e gravar em arquivo.

In [ ]:
# @title 🖥️ Painel resumido da GPU (real ou simulado)
# ============================================================================
# OBJETIVO: mostrar os campos essenciais da GPU de forma legível.
# Com GPU real, usamos o nvidia-smi; sem GPU, imprimimos um exemplo simulado.
# ============================================================================
def painel_gpu():
    if TEM_GPU:
        # Pede só os campos que interessam, em CSV sem cabeçalho e sem unidades
        saida = subprocess.run(
            ["nvidia-smi",
             "--query-gpu=index,name,temperature.gpu,utilization.gpu,"
             "memory.used,memory.total,power.draw",
             "--format=csv,noheader,nounits"],
            capture_output=True, text=True,
        ).stdout.strip()
        print("GPU(s) detectada(s) — index, nome, temp, util, VRAM usada/total, potência:")
        print(saida)
    else:
        # Exemplo no MESMO formato, para a aula continuar sem GPU
        print("Sem GPU real — exemplo simulado (Tesla T4):")
        print("0, Tesla T4, 52, 87, 3842, 15360, 48.2")

painel_gpu()

## 5. Atividade: somar muitos números — sequencial vs. vetorizado

**Pergunta:** se cada instrução leva 1 ciclo, quanto tempo leva para somar 1000 números?

- **CPU (1 núcleo):** 1000 × 1 ciclo = **1000 ciclos**
- **GPU (1000 núcleos):** 1000 ÷ 1000 = **1 ciclo** → ≈ **1000× mais rápido**

Vamos medir isso com **multiplicação de matrizes**, a operação central da IA:
uma versão **sequencial** (3 `for` aninhados) contra a **vetorizada** (NumPy/BLAS, que
explora o paralelismo do hardware).

In [ ]:
# @title ⏱️ Benchmark: 3 loops (sequencial) vs. NumPy (vetorizado)
# ============================================================================
# OBJETIVO: medir, na prática, o ganho do paralelismo de hardware.
# A versão sequencial faz um cálculo por vez (como a CPU em um núcleo);
# a versão NumPy delega para bibliotecas otimizadas (BLAS), que usam
# múltiplos núcleos e instruções SIMD — o espírito da GPU.
# ============================================================================
import time
import numpy as np

N = 200  # tamanho da matriz N x N (aumente para ver o efeito crescer)

# Cria duas matrizes aleatórias em precisão simples (float32, como na GPU)
A = np.random.rand(N, N).astype(np.float32)
B = np.random.rand(N, N).astype(np.float32)

print(f"Multiplicação de matrizes {N}x{N}")
print(f"Total de operações: {N**3 * 2:,} (multiply-add)\n")

# ── Versão 1: sequencial (3 loops aninhados) ────────────────────────────────
inicio = time.time()
C = np.zeros((N, N), dtype=np.float32)
for i in range(N):            # percorre as linhas
    for j in range(N):        # percorre as colunas
        soma = 0.0
        for k in range(N):    # soma os produtos (um por vez)
            soma += A[i, k] * B[k, j]
        C[i, j] = soma
tempo_seq = time.time() - inicio
print(f"Sequencial (3 loops): {tempo_seq:.2f}s")

# ── Versão 2: vetorizada (NumPy usa BLAS paralelo) ──────────────────────────
inicio = time.time()
C_np = A @ B                  # o operador @ faz a multiplicação otimizada
tempo_np = time.time() - inicio
print(f"Vetorizado (NumPy):   {tempo_np:.6f}s")

# ── Validação: as duas versões devem dar o mesmo resultado ──────────────────
assert np.allclose(C, C_np, atol=1e-2), "Resultados divergem!"
print(f"\n✓ Resultados conferem — Speedup: {tempo_seq / tempo_np:.0f}x")
print("→ Em redes neurais, essas matrizes têm milhões de linhas. Sem paralelismo, é inviável.")

### O que acabamos de ver

A mesma conta, feita de duas formas, produz o **mesmo resultado** — mas a versão
vetorizada é **muito** mais rápida, porque o hardware processa vários elementos ao
mesmo tempo. Isso é o embrião do que a GPU faz em escala gigante: **paralelismo massivo**.

## 6. Discussão em Grupo

Em grupos de 3–4, discutam:

1. Em que aplicações a **CPU** ainda é melhor que a GPU?
2. Por que um **smartphone** usa GPU integrada?
3. Vale a pena usar GPU para um **site simples**? Por quê?
4. Como **escolher entre CPU e GPU** para um projeto de IA?

> Atividade de pesquisa completa em `aulas/aula01/atividade.md`.

## 7. Síntese e Tarefa de Casa

**O que levar:**
- **Von Neumann** — memória compartilhada; maioria dos PCs.
- **Harvard** — memórias separadas; sistemas embarcados.
- **Gargalo de Von Neumann** — barramento único limita o ritmo.
- **GPU** — paralelismo massivo; IA, renderização e simulações.

**Tarefa (opcional):** encontre um caso real em que a GPU acelerou uma aplicação de IA
(Tesla Autopilot, diagnóstico médico, ChatGPT, renderização de filmes) e registre o que a
solução faz, qual GPU usa e o ganho relatado.

> 🔗 **Próxima aula:** *Modelos de Processamento* — como a GPU executa tudo isso em
> paralelo (SIMD, MIMD, RISC e CISC).